# Band Gap and Density of States of an H-Passivated Si Nanowire

> **B. Aradi, L. E. Ramos, P. Deák, Th. Köhler, F. Bechstedt, R. Q. Zhang, and Th. Frauenheim**
> "Theoretical study of the chemical gap tuning in silicon nanowires"
> Phys. Rev. B 76, 035305 (2007)
> [DOI: 10.1103/PhysRevB.76.035305](https://doi.org/10.1103/PhysRevB.76.035305)

Computes the band gap, the bands along the wire and the density of states of the H-passivated ⟨110⟩ silicon nanowire
of radius 0.51 nm, one period Si₁₈H₁₂ as created in the [structure notebook](passivation_edge_nanowire_silicon.ipynb),
and the band gap of bulk silicon, and compares them with Aradi et al.'s Table II and
Fig. 8 (left). The manuscript optimizes the wire with DFTB and computes its gaps with CRYSTAL and a one-parameter global
hybrid fitted to bulk silicon (1.16 eV); here the atoms and the period of the wire are relaxed with PBE and the gaps are
computed with a global hybrid in the PBE0 form, its fraction of exact exchange (cell 1.3) fitted the same way.

<h2 style="color:green">Usage</h2>

1. Create the material in the [structure notebook](passivation_edge_nanowire_silicon.ipynb), which saves `Si nanowire 110 H-passivated one period` to the `uploads` folder.
1. Set the parameters in cells 1.2 and 1.3 (or use the defaults).
1. Click "Run" > "Run All" to run all cells.
1. Wait for the jobs to complete.
1. Scroll down to view the results.

## Summary

1. Set up the environment and parameters: install packages (JupyterLite only) and configure the material names, compute, DFT and plot parameters.
1. Authenticate and initialize API client: authenticate via browser, initialize the client, then select account and project.
1. Load the wire by name and bulk silicon from Standata, print provenance and save both to the platform.
1. Configure the workflows: variable-cell relaxation of the wire, then, with the global hybrid, band structure of the wire, band gap and density of states of the wire and of bulk silicon.
1. Configure compute: get the list of clusters and create a compute configuration.
1. Run the jobs: bulk silicon and the relaxation of the wire, then the band structure and the density of states of the relaxed wire.
1. Retrieve the results: the period of the relaxed wire and the band gaps.
1. Compare with Aradi et al. (2007): the band gaps, then the bands and the density of states beside Fig. 8 (left).

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters and configurations for the workflows and jobs

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

ORGANIZATION_NAME = None  # set to your organization name (full or partial); otherwise, your default one is used
FOLDER = "./uploads"
WIRE_NAME = "Si nanowire 110 H-passivated one period"  # saved by the structure notebook
RELAXED_WIRE_NAME = "Si nanowire 110 H-passivated relaxed one period"
BULK_NAME = "Silicon"  # Standata

APPLICATION_NAME = "espresso"
RELAX_WORKFLOW_SEARCH_TERM = "variable_cell_relaxation.json"
BANDS_WORKFLOW_SEARCH_TERM = "band_structure_hse.json"
DOS_WORKFLOW_SEARCH_TERM = "band_gap_dos_hse.json"

CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.OF
PPN = None  # None uses the selected queue's maximum cores per job on the selected cluster
NPOOLS = 2  # k-point pools: must divide the cores and not exceed any job's k-points; each adds exact-exchange memory
TIME_LIMIT = "24:00:00"

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 60  # seconds

ENERGY_WINDOW = (-4.0, 6.0)  # eV, the energy range of Fig. 8

### 1.3. Set the DFT parameters

In [ ]:
# manuscript: geometry with DFTB, gaps with CRYSTAL and a one-parameter global hybrid fitted to bulk Si;
# ours: geometry with PBE, GBRV ultrasoft pseudopotentials at their tested cutoffs; gaps with that hybrid in the
# PBE0 form and PseudoDojo norm-conserving ones, as ultrasoft augmentation of every band pair dominates its cost
ECUTWFC = 40  # Ry
ECUTRHO = 200  # Ry
HSE_PSEUDOPOTENTIAL_TYPE = "nc"
HSE_PSEUDOPOTENTIAL_SEARCH_TEXT = "dojo-oncv/0.4"  # PseudoDojo 0.4 among norm-conserving files for hybrid functionals
HSE_ECUTWFC = 72  # Ry, PseudoDojo "normal" hint: H 36 Ha, Si 18 Ha
HSE_ECUTRHO = 4 * HSE_ECUTWFC  # Ry, enough for norm-conserving pseudopotentials
# fraction of exact exchange, fitted (to two decimals) so bulk Si gives the manuscript's 1.16 eV; section 8.1 prints it
EXX_FRACTION = 0.10

# forces below 1e-4 Ha/Bohr as in the manuscript, in QE's Ry/Bohr; the atoms and the period along the wire relax
RELAXATION_SETTINGS = {"control": {"forc_conv_thr": 2e-4, "nstep": 100}, "cell": {"cell_dofree": "z"}}
PRELIMINARY_SCF_SETTINGS = {"control": {"verbosity": "high"}}  # so pw.x prints the crystal k-points the hybrid run uses
# Quantum ESPRESSO computes no hybrid-functional stress with pools; ecutfock (Ry) as in the DOS workflow, since
# its default, 4 x HSE_ECUTWFC, would multiply each pool's exchange memory
HSE_SETTINGS = {"control": {"tstress": False},
                "system": {"input_dft": "pbe0", "exx_fraction": EXX_FRACTION, "ecutfock": 100}}
RELAX_KGRID = [1, 1, 8]  # manuscript: 1x1x4 on two periods
BANDS_KGRID = [1, 1, 8]
QGRID = [1, 1, 4]  # exact-exchange q-grid; each dimension divides the one of BANDS_KGRID
KPATH_STEPS = 20  # even, so every k − q of the path stays on the path for the 1x1x4 q-grid
# a job is reused by material, workflow name and k-grid, so rename a workflow after changing any other setting it uses
RELAX_WORKFLOW_NAME = "Variable-cell Relaxation Si nanowire"
BANDS_WORKFLOW_NAME = f"Band Structure global hybrid {EXX_FRACTION} Si nanowire"
WIRE_DOS_WORKFLOW_NAME = f"Band Gap + DOS global hybrid {EXX_FRACTION} Si nanowire"
BULK_DOS_WORKFLOW_NAME = f"Band Gap + DOS global hybrid {EXX_FRACTION} Si bulk"
DOS_KGRID = [1, 1, 8]  # the workflow runs odd dimensions as the next even, so 2x2x8
BULK_KGRID = [8, 8, 8]

RELAX_UNIT = "pw_vc-relax"
PRELIMINARY_SCF_UNIT = "pw_scf"
BANDS_HSE_UNIT = "pw_scf_bands_hse"
BANDS_UNIT = "bands"
DOS_HSE_UNIT = "pw_scf_hse"

## 2. Authenticate and initialize API client
### 2.1. Authenticate
Authenticate in the browser and have credentials stored in environment variable "OIDC_ACCESS_TOKEN".

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.2. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.3. Select account

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.4. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load the materials
### 3.1. Load the wire from the uploads folder or the platform, bulk silicon from Standata, and print provenance

In [ ]:
from collections import Counter
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.core.entity.material.api import load_material
from mat3ra.standata.materials import Materials

materials = {
    WIRE_NAME: load_material(client, FOLDER, WIRE_NAME, ACCOUNT_ID),
    BULK_NAME: Material.create(Materials.get_by_name_first_match(BULK_NAME)),
}
for name, material in materials.items():
    print(f"{name}: {dict(Counter(material.basis.elements.values))}, {material.basis.number_of_atoms} atoms, "
          f"cell {material.lattice.a:.3f} x {material.lattice.b:.3f} x {material.lattice.c:.3f} Å")

### 3.2. Save the materials to the platform

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material

saved_materials = {name: get_or_create_material(client, material, ACCOUNT_ID) for name, material in materials.items()}

## 4. Configure the workflows

The Standata workflows carry their models: PBE for the relaxation and for the preliminary SCF of the band structure,
HSE06 for the band structure and for the band gap and density of states, all with GBRV ultrasoft pseudopotentials;
4.2 and 4.3 switch those runs to norm-conserving ones and set the global hybrid in their input files.
### 4.1. Variable-cell relaxation of the wire

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.notebooks_utils.workflow import apply_planewave_cutoffs, apply_scf_kgrid, patch_workflow_qe_input

workflow_configs = WorkflowStandata.filter_by_application(APPLICATION_NAME)
wire = materials[WIRE_NAME]

relax_workflow = Workflow.create(workflow_configs.get_by_name_first_match(RELAX_WORKFLOW_SEARCH_TERM))
relax_workflow.name = RELAX_WORKFLOW_NAME
apply_planewave_cutoffs(relax_workflow, ECUTWFC, ECUTRHO, unit_name=RELAX_UNIT)
apply_scf_kgrid(relax_workflow, RELAX_KGRID, material=wire, unit_name=RELAX_UNIT)
patch_workflow_qe_input(relax_workflow, RELAXATION_SETTINGS, [RELAX_UNIT])

### 4.2. Global-hybrid band structure of the wire

The PBE SCF on `BANDS_KGRID` gives the k-points of the hybrid run, which adds the path along the wire (6.2) and
evaluates the exchange on `QGRID`.

In [ ]:
from mat3ra.mode import ModelFactory
from mat3ra.standata.model_tree import ModelTreeStandata
from mat3ra.wode.context.providers import PointsGridDataProvider

bands_workflow = Workflow.create(workflow_configs.get_by_name_first_match(BANDS_WORKFLOW_SEARCH_TERM))
bands_workflow.name = BANDS_WORKFLOW_NAME
apply_scf_kgrid(bands_workflow, BANDS_KGRID, material=wire, unit_name=PRELIMINARY_SCF_UNIT)
apply_planewave_cutoffs(bands_workflow, ECUTWFC, ECUTRHO, unit_name=PRELIMINARY_SCF_UNIT)
apply_planewave_cutoffs(bands_workflow, HSE_ECUTWFC, HSE_ECUTRHO, unit_name=BANDS_HSE_UNIT)
patch_workflow_qe_input(bands_workflow, PRELIMINARY_SCF_SETTINGS, [PRELIMINARY_SCF_UNIT])
patch_workflow_qe_input(bands_workflow, HSE_SETTINGS, [BANDS_HSE_UNIT])
main_hse_subworkflow = next(
    subworkflow for subworkflow in bands_workflow.subworkflows if subworkflow.name == "Main HSE Run"
)
hse_model_config = ModelTreeStandata.get_model_by_parameters(
    type="dft", subtype=main_hse_subworkflow.model.subtype, functional=main_hse_subworkflow.model.functional
)
hse_model_config["method"] = {
    "type": "pseudopotential",
    "subtype": HSE_PSEUDOPOTENTIAL_TYPE,
    "data": {"searchText": HSE_PSEUDOPOTENTIAL_SEARCH_TEXT},
}
main_hse_subworkflow.model = ModelFactory.create(hse_model_config)
bands_hse_unit = main_hse_subworkflow.get_unit_by_name(name=BANDS_HSE_UNIT)
bands_hse_unit.add_context(
    PointsGridDataProvider(name="qgrid", material=wire, dimensions=QGRID, isEdited=True).get_context_item_data()
)
main_hse_subworkflow.set_unit(bands_hse_unit)
bands_unit = main_hse_subworkflow.get_unit_by_name(name=BANDS_UNIT)
bands_unit.results = [{"name": "band_structure"}, {"name": "band_gaps"}]
main_hse_subworkflow.set_unit(bands_unit)

### 4.3. Global-hybrid band gap and density of states of the wire and of bulk silicon

In [ ]:
dos_workflows = {}
for name, kgrid, workflow_name in ((WIRE_NAME, DOS_KGRID, WIRE_DOS_WORKFLOW_NAME),
                                   (BULK_NAME, BULK_KGRID, BULK_DOS_WORKFLOW_NAME)):
    workflow = Workflow.create(workflow_configs.get_by_name_first_match(DOS_WORKFLOW_SEARCH_TERM))
    workflow.name = workflow_name
    apply_planewave_cutoffs(workflow, HSE_ECUTWFC, HSE_ECUTRHO, unit_name=DOS_HSE_UNIT)
    apply_scf_kgrid(workflow, kgrid, material=materials[name], unit_name=DOS_HSE_UNIT)
    patch_workflow_qe_input(workflow, HSE_SETTINGS, [DOS_HSE_UNIT])
    workflow.subworkflows[0].model = ModelFactory.create(hse_model_config)
    dos_workflows[name] = workflow

## 5. Create the compute configuration
### 5.1. Get list of clusters

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create the compute configuration for the jobs

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
    if cluster is None:
        raise ValueError(f"Cluster '{CLUSTER_NAME}' not found. Available: {[c['hostname'] for c in clusters]}")
else:
    cluster = clusters[0]
ppn = PPN or next(queue["maxPPN"] for queue in cluster["queues"] if queue["name"] == QUEUE_NAME.value)
compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=ppn, timeLimit=TIME_LIMIT, arguments={"npools": NPOOLS})
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {ppn}, "
      f"time limit: {TIME_LIMIT}")

## 6. Run the jobs

A submitted, queued, running or finished job for the same material, workflow name and k-grid is reused.
### 6.1. Bulk silicon and the relaxation of the wire

In [ ]:
from mat3ra.notebooks_utils.core.entity.job.api import find_job_for_material
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.api.job import submit_jobs, wait_for_jobs_to_finish_async

job_ids = {}
for workflow, saved_material, kgrid, unit_name in (
    (dos_workflows[BULK_NAME], saved_materials[BULK_NAME], BULK_KGRID, DOS_HSE_UNIT),
    (relax_workflow, saved_materials[WIRE_NAME], RELAX_KGRID, RELAX_UNIT),
):
    job = find_job_for_material(client, saved_material["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"),
                                kgrid=kgrid, unit_name=unit_name)
    if job is None:
        job = create_job(api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
                         owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}")
        submit_jobs(client.jobs, [job["_id"]])
        print(f"✅ {workflow.name}: submitted job {job['_id']}")
    else:
        print(f"♻️  {workflow.name}: reusing job {job['_id']}")
    job_ids[workflow.name] = job["_id"]

In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, [job_ids[relax_workflow.name]], poll_interval=POLL_INTERVAL)

### 6.2. Relaxed wire and the path along it

The end of the Brillouin zone along the wire, (0, 0, 1/2), is Z in the manuscript; its name in the k-path depends on
the lattice type of the relaxed cell, so it is looked up by its coordinates.

In [ ]:
from mat3ra.made.reciprocal.symmetry_points import get_symmetry_points
from mat3ra.notebooks_utils.core.entity.material.api import get_final_structure_for_job
from mat3ra.wode.context.providers import PointsPathDataProvider

relaxed_wire = get_final_structure_for_job(client, job_ids[relax_workflow.name])
client.materials.update(relaxed_wire.id, {"name": RELAXED_WIRE_NAME})
wire_axis_point = next(
    point["point"] for point in get_symmetry_points(relaxed_wire.lattice) if point["coordinates"] == [0, 0, 0.5]
)
bands_hse_unit.add_context(
    PointsPathDataProvider(
        name="explicitKPath",
        path=[{"point": "Γ", "steps": KPATH_STEPS}, {"point": wire_axis_point, "steps": 1}],
        isEdited=True,
    ).get_context_item_data()
)
main_hse_subworkflow.set_unit(bands_hse_unit)

### 6.3. Band structure and density of states of the relaxed wire

In [ ]:
for workflow, saved_material, kgrid, unit_name in (
    (bands_workflow, relaxed_wire.to_dict(), BANDS_KGRID, PRELIMINARY_SCF_UNIT),
    (dos_workflows[WIRE_NAME], relaxed_wire.to_dict(), DOS_KGRID, DOS_HSE_UNIT),
):
    job = find_job_for_material(client, saved_material["_id"], workflow.name, ACCOUNT_ID,
                                statuses=("submitted", "queued", "active", "finished"),
                                kgrid=kgrid, unit_name=unit_name)
    if job is None:
        job = create_job(api_client=client, materials=[saved_material], workflow=workflow, project_id=project_id,
                         owner_id=ACCOUNT_ID, compute=compute.to_dict(), prefix=f"{workflow.name} {timestamp}")
        submit_jobs(client.jobs, [job["_id"]])
        print(f"✅ {workflow.name}: submitted job {job['_id']}")
    else:
        print(f"♻️  {workflow.name}: reusing job {job['_id']}")
    job_ids[workflow.name] = job["_id"]

In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, list(job_ids.values()), poll_interval=POLL_INTERVAL)

## 7. Retrieve the results
### 7.1. Period of the relaxed wire

In [ ]:
print(f"Period along the wire, {WIRE_NAME}:")
print(f"start: {wire.lattice.c:.4f} Å")
print(f"relaxed (PBE): {relaxed_wire.lattice.c:.4f} Å")

### 7.2. Band gaps

The fundamental gap is the `indirect` entry of a job's band gaps: from the valence-band maximum to the conduction-band
minimum wherever they fall, direct when both are at the same k-point. The gap of the wire comes from the band structure
job, whose k-points include the path along the wire.

In [ ]:
from mat3ra.notebooks_utils.core.entity.property.api import get_properties_for_job

fundamental_gaps = {}
for workflow in (bands_workflow, dos_workflows[WIRE_NAME], dos_workflows[BULK_NAME]):
    band_gaps = get_properties_for_job(client, job_ids[workflow.name], "band_gaps")[0]
    fundamental_gaps[workflow.name] = next(gap for gap in band_gaps["values"] if gap["type"] == "indirect")

## 8. Compare with Aradi et al. (2007)
### 8.1. Band gaps

In [ ]:
# Aradi et al. 2007: Table II (p. 5) for the wire; bulk Si with the same hybrid, p. 6
MANUSCRIPT_BAND_GAPS = {WIRE_NAME: (2.21, "direct"), BULK_NAME: (1.16, "indirect")}  # eV

print(f"Band gap (global hybrid, exact-exchange fraction {EXX_FRACTION}):")
for label, gap in ((WIRE_NAME, fundamental_gaps[bands_workflow.name]),
                   (BULK_NAME, fundamental_gaps[dos_workflows[BULK_NAME].name])):
    character = "direct" if gap["kpointValence"] == gap["kpointConduction"] else "indirect"
    print(f"{label}: {gap['value']:.2f} eV, {character}, VBM at k = {tuple(gap['kpointValence'])}, "
          f"CBM at k = {tuple(gap['kpointConduction'])}")

print("\nBand gap from the manuscript:")
for label, (value, character) in MANUSCRIPT_BAND_GAPS.items():
    print(f"{label}: {value:.2f} eV, {character}")

### 8.2. Bands and density of states (Fig. 8 of the manuscript, left)

The manuscript's Fig. 8 (left): density of states (solid), its projection on the H atoms (dotted) and bands along Γ–Z,
the valence-band maximum at 0:

<img src="https://github.com/Exabyte-io/documentation/raw/38c12d16dda9636d07caea4ab7b1c40602d4e8f9/images/tutorials/materials/passivation/passivation_edge_nanowire_silicon/aradi2007-fig8-left-bands-dos.webp" alt="Aradi et al. (2007), Fig. 8 (left): density of states and bands along Γ–Z of the H-passivated ⟨110⟩ Si nanowire, r = 0.51 nm" width="400"/>

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from mat3ra.notebooks_utils.plot import display_matplotlib_figure

band_structure = get_properties_for_job(client, job_ids[bands_workflow.name], "band_structure")[0]
density_of_states = get_properties_for_job(client, job_ids[dos_workflows[WIRE_NAME].name], "density_of_states")[0]
path_bands = np.array(band_structure["yDataSeries"]) - fundamental_gaps[bands_workflow.name]["eigenvalueValence"]
last_path_index = len(band_structure["xDataArray"]) - 1
dos_energies = np.array(density_of_states["xDataArray"][0])
dos_energies -= fundamental_gaps[dos_workflows[WIRE_NAME].name]["eigenvalueValence"]
hydrogen_dos = np.sum([series for series, legend in zip(density_of_states["yDataSeries"], density_of_states["legend"])
                       if legend.get("element") == "H"], axis=0)

figure, (dos_axes, bands_axes) = plt.subplots(1, 2, sharey=True, gridspec_kw={"width_ratios": [1, 2], "wspace": 0})
dos_axes.plot(density_of_states["yDataSeries"][0], dos_energies, color="black", label="total")
dos_axes.plot(hydrogen_dos, dos_energies, color="black", linestyle=":", label="H")
dos_axes.invert_xaxis()
dos_axes.set_xticks([])
dos_axes.set_ylabel("Energy (eV)")
dos_axes.legend()
bands_axes.plot(path_bands.T, color="black", linewidth=0.8)
bands_axes.set_xlim(0, last_path_index)
bands_axes.set_xticks([0, last_path_index], ["Γ", "Z"])
bands_axes.set_ylim(*ENERGY_WINDOW)
figure.suptitle(f"{WIRE_NAME}\nglobal hybrid, exact-exchange fraction {EXX_FRACTION}")
display_matplotlib_figure(figure)

## References

[1] B. Aradi, L. E. Ramos, P. Deák, Th. Köhler, F. Bechstedt, R. Q. Zhang and Th. Frauenheim, "Theoretical study of the chemical gap tuning in silicon nanowires", Phys. Rev. B 76, 035305 (2007). https://doi.org/10.1103/PhysRevB.76.035305

[2] J. P. Perdew, M. Ernzerhof and K. Burke, "Rationale for mixing exact exchange with density functional approximations", J. Chem. Phys. 105, 9982 (1996). https://doi.org/10.1063/1.472933

[3] C. Adamo and V. Barone, "Toward reliable density functional methods without adjustable parameters: The PBE0 model", J. Chem. Phys. 110, 6158 (1999). https://doi.org/10.1063/1.478522

[4] K. F. Garrity, J. W. Bennett, K. M. Rabe and D. Vanderbilt, "Pseudopotentials for high-throughput DFT calculations", Comput. Mater. Sci. 81, 446 (2014). https://doi.org/10.1016/j.commatsci.2013.08.053

[5] M. J. van Setten, M. Giantomassi, E. Bousquet, M. J. Verstraete, D. R. Hamann, X. Gonze and G.-M. Rignanese, "The PseudoDojo: Training and grading a 85 element optimized norm-conserving pseudopotential table", Comput. Phys. Commun. 226, 39 (2018). https://doi.org/10.1016/j.cpc.2018.01.012